In [ ]:
import json
import pandas as pd
import re
import os

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [ ]:
# Pegar so os links do jsonl
file_path = "youtube_links.jsonl"

video_links = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        url = data["url"]
        
        
        match = re.search(r"v=([A-Za-z0-9_\-]{11})", url)
        if match:
            video_id = match.group(1)
            clean_url = f"https://www.youtube.com/watch?v={video_id}"
            video_links.append(clean_url)

df = pd.DataFrame(video_links, columns=["youtube_video_link"])


print(df.head())
print(f"\nTotal de links de vídeo limpos: {len(df)}")


In [ ]:
# Pegar os titulos dos videos dos links
api_keys = [
    "Chave Api 1",
    "Chave Api 2" # Pode por quantas quiser 
]

file_path = "youtube_links.jsonl"
output_csv_path = "youtube_titulos_saida.csv" 


def get_all_video_ids(file_path):
    video_ids = set()
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            url = data["url"]
            match = re.search(r"v=([A-Za-z0-9_\-]{11})", url)
            if match:
                video_ids.add(match.group(1))
    return list(video_ids)

def get_processed_ids(csv_path):
    if not os.path.exists(csv_path):
        return set()
    df_existing = pd.read_csv(csv_path)
    if 'video_id' in df_existing.columns:
        return set(df_existing['video_id'].dropna())
    return set()


# Carrega todos os IDs e filtra os que já foram processados
all_ids = get_all_video_ids(file_path)
processed_ids = get_processed_ids(output_csv_path)
ids_to_process = [vid_id for vid_id in all_ids if vid_id not in processed_ids]

print(f"Encontrados {len(all_ids)} IDs. Já processados: {len(processed_ids)}. Restam: {len(ids_to_process)}")

if not ids_to_process:
    print("Nenhum vídeo novo para processar.")
else:
    # Usa a primeira key
    current_key_index = 0
    youtube = build("youtube", "v3", developerKey=api_keys[current_key_index])
    print(f"Iniciando com a chave API #{current_key_index + 1}")

    results = []
    # Processa de 50 em 50 
    for i in range(0, len(ids_to_process), 50):
        batch_ids = ids_to_process[i:i + 50]
        
        try:
            request = youtube.videos().list(
                part="snippet",
                id=",".join(batch_ids)
            )
            response = request.execute()
            
            for item in response.get("items", []):
                results.append({
                    "video_id": item["id"],
                    "title": item["snippet"]["title"],
                    "youtube_video_link": f'https://www.youtube.com/watch?v={item["id"]}'
                })

        except HttpError as e:
            # Se der erro, troca a chave
            if e.resp.status == 403:
                print(f"Quota da chave #{current_key_index + 1} esgotada.")
                current_key_index += 1 # Avança para o próximo índice de chave

                if current_key_index < len(api_keys):
                    print(f"TROCANDO para a chave API #{current_key_index + 1}.")

                    youtube = build("youtube", "v3", developerKey=api_keys[current_key_index])
                    try:
                        request = youtube.videos().list(
                            part="snippet",
                            id=",".join(batch_ids)
                        )
                        response = request.execute()
                        for item in response.get("items", []):
                             results.append({
                                "video_id": item["id"],
                                "title": item["snippet"]["title"],
                                "youtube_video_link": f'https://www.youtube.com/watch?v={item["id"]}'
                            })
                    except Exception as inner_e:
                        print(f"Erro ao tentar novamente com a nova chave: {inner_e}")

                else:
                    print("Todas as chaves de API foram esgotadas. Parando o processo.")
                    break 
            else:
                print(f"Ocorreu um erro na API: {e}")
        
        print(f"Progresso: {len(processed_ids) + len(results)} / {len(all_ids)} vídeos")


    # Salva o resultado e gera o .csv
    if results:
        df_new = pd.DataFrame(results)
        if os.path.exists(output_csv_path):
            df_new.to_csv(output_csv_path, mode='a', header=False, index=False, encoding='utf-8')
        else:
            df_new.to_csv(output_csv_path, index=False, encoding='utf-8')
        print(f"Processamento concluído. {len(results)} novos vídeos salvos em '{output_csv_path}'.")
    else:
        print("Nenhum resultado novo foi gerado.")

In [ ]:
pd.set_option('display.max_colwidth', None)
df_final = pd.read_csv('youtube_titulos_saida.csv')

print(f"Dimensões do DataFrame: {df_final.shape}")

df_final.head().style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

In [ ]:
# Célula pra testar se a API Key ta funcionando
CHAVE_API_PARA_TESTAR = "COLE_AQUI_SUA_CHAVE_MAIS_NOVA"

print(f"Tentando usar a chave que termina em: '...{CHAVE_API_PARA_TESTAR[-4:]}'")

try:
    youtube = build('youtube', 'v3', developerKey=CHAVE_API_PARA_TESTAR)
    
    request = youtube.videos().list(
        part="snippet",
        id="dQw4w9WgXcQ" 
    )
    response = request.execute()

    print("\n✅ SUCESSO! A chave de API está funcionando corretamente.")
    print(f"Título do vídeo encontrado: {response['items'][0]['snippet']['title']}")

except HttpError as e:
    print("\n❌ FALHA! A chave não funcionou.")
    print("--- MENSAGEM DE ERRO DETALHADA ---")
    print(e)
    print("------------------------------------")

except Exception as e:
    print(f"\n❌ FALHA! Ocorreu um erro inesperado: {e}")